In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load from CSVs we saved earlier
gold_movies    = pd.read_csv('../data/final/gold_movies.csv')
genre_matrix   = pd.read_csv('../data/final/genre_matrix.csv')

print("✅ Gold data loaded!")
print("Movies:", gold_movies.shape)
print("Genre Matrix:", genre_matrix.shape)

✅ Gold data loaded!
Movies: (9708, 7)
Genre Matrix: (9708, 21)


In [2]:
# Use only genre columns for similarity (drop movieId and clean_title)
genre_cols = genre_matrix.drop(columns=['movieId', 'clean_title'])

# Calculate cosine similarity between all movies
similarity = cosine_similarity(genre_cols)

# Convert to DataFrame for easy lookup
similarity_df = pd.DataFrame(
    similarity,
    index=genre_matrix['clean_title'],
    columns=genre_matrix['clean_title']
)

print("✅ Similarity matrix created!")
print("Shape:", similarity_df.shape)

✅ Similarity matrix created!
Shape: (9708, 9708)


In [3]:
def recommend_movies(movie_title, top_n=10):
    """
    Given a movie title, return top N similar movies
    """
    # Check if movie exists
    if movie_title not in similarity_df.index:
        return f"❌ Movie '{movie_title}' not found!"
    
    # Get similarity scores for this movie
    scores = similarity_df[movie_title].sort_values(ascending=False)
    
    # Remove the movie itself (it's always #1 match)
    scores = scores.drop(movie_title, errors='ignore')
    
    # Get top N similar movie titles
    top_titles = scores.head(top_n).index.tolist()
    
    # Get full details from gold_movies
    recommendations = gold_movies[
        gold_movies['clean_title'].isin(top_titles)
    ][['clean_title', 'genres', 'avg_rating', 'num_ratings', 'year']]
    
    recommendations = recommendations.sort_values('avg_rating', ascending=False)
    
    return recommendations

print("✅ Recommendation function ready!")

✅ Recommendation function ready!


In [4]:
# Test with a popular movie
results = recommend_movies("Toy Story", top_n=10)
print("🎬 Because you liked Toy Story, you might also like:\n")
print(results.to_string(index=False))

🎬 Because you liked Toy Story, you might also like:

                            clean_title                                          genres  avg_rating  num_ratings   year
                         Monsters, Inc. Adventure, Animation, Children, Comedy, Fantasy        3.87        132.0 2001.0
                            Toy Story 2 Adventure, Animation, Children, Comedy, Fantasy        3.86         97.0 1999.0
              Emperor's New Groove, The Adventure, Animation, Children, Comedy, Fantasy        3.72         37.0 2000.0
                                  Moana Adventure, Animation, Children, Comedy, Fantasy        3.45         10.0 2016.0
                                   Antz Adventure, Animation, Children, Comedy, Fantasy        3.24         45.0 1998.0
                Tale of Despereaux, The Adventure, Animation, Children, Comedy, Fantasy        3.00          1.0 2008.0
                      The Good Dinosaur Adventure, Animation, Children, Comedy, Fantasy        3.00        

In [5]:
print("🎬 Recommendations for Forrest Gump:\n")
print(recommend_movies("Forrest Gump").to_string(index=False))

print("\n🎬 Recommendations for The Dark Knight:\n")
print(recommend_movies("Dark Knight, The").to_string(index=False))

🎬 Recommendations for Forrest Gump:

                                                    clean_title                      genres  avg_rating  num_ratings   year
                                                       3 Idiots      Comedy, Drama, Romance        4.75          2.0 2009.0
                                   Train of Life (Train de vie) Comedy, Drama, Romance, War        4.50          2.0 1998.0
I Served the King of England (Obsluhoval jsem anglického krále) Comedy, Drama, Romance, War        4.50          1.0 2006.0
                            Life Is Beautiful (La Vita è bella) Comedy, Drama, Romance, War        4.15         88.0 1997.0
                                            Lost in Translation      Comedy, Drama, Romance        4.03         74.0 2003.0
                   Tiger and the Snow, The (La tigre e la neve) Comedy, Drama, Romance, War        4.00          1.0 2005.0
                                                          Dummy      Comedy, Drama, Romance    

In [7]:
import pickle

# Save similarity matrix and movie list for Streamlit app
with open('../data/final/similarity.pkl', 'wb') as f:
    pickle.dump(similarity_df, f)

gold_movies.to_csv('../data/final/gold_movies.csv', index=False)

print("✅ Model saved!")
print("📦 similarity.pkl → ready for Streamlit")

✅ Model saved!
📦 similarity.pkl → ready for Streamlit
